# CarePath - Use a Shared GEC Adapter

For someone who received a trained CarePath adapter (a Google Drive folder or
.zip) and wants to run it. **You need a GPU** (Colab T4/L4, or a local NVIDIA
GPU). You do **not** need the dataset, the repo, or to train anything - just the
adapter folder. The base model downloads automatically on first use.

1. Install.  2. Point at the adapter.  3. Run - see raw ASR vs corrected text.

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes sentencepiece gdown

## Get the adapter the trainer shared

In [ ]:
# --- Point this at the adapter the trainer shared (pick ONE) ---
# A) Colab + Drive: trainer shared a folder. In Drive 'Add shortcut to Drive',
#    then set ADAPTER_DIR to its mounted path under /content/drive/MyDrive/...
# B) Local: the folder you downloaded and unzipped.
# C) Any: paste a Google Drive *share link* to the .zip; gdown fetches + unzips.
ADAPTER_DIR = ''        # e.g. '/content/drive/MyDrive/carepath-gec-qwen3-full' or './carepath-gec-qwen3-full'
ADAPTER_ZIP_URL = ''    # OR a Google Drive share link to the .zip

import importlib.util, subprocess, sys, zipfile
from pathlib import Path
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

if ADAPTER_ZIP_URL:
    import gdown
    gdown.download(url=ADAPTER_ZIP_URL, output='adapter.zip', fuzzy=True)
    with zipfile.ZipFile('adapter.zip') as z:
        z.extractall('downloaded_adapter')
    cfg = next(Path('downloaded_adapter').rglob('adapter_config.json'), None)
    if cfg is None:
        raise SystemExit('That zip has no adapter_config.json inside.')
    ADAPTER_DIR = str(cfg.parent)
elif IN_COLAB and ADAPTER_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

if not ADAPTER_DIR or not (Path(ADAPTER_DIR) / 'adapter_config.json').exists():
    raise SystemExit('Set ADAPTER_DIR to the unzipped adapter folder (with adapter_config.json), '
                     'or set ADAPTER_ZIP_URL to a Drive share link.')
print('Using adapter:', ADAPTER_DIR)

## Load it and try a few transcripts

In [ ]:
import json
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Use a Colab GPU runtime or a local NVIDIA GPU.')

# The adapter records which base model it was trained on - use that exact one.
acfg = json.loads((Path(ADAPTER_DIR) / 'adapter_config.json').read_text(encoding='utf-8'))
BASE_MODEL = acfg.get('base_model_name_or_path') or 'Qwen/Qwen3-4B-Instruct-2507'
vfile = Path(ADAPTER_DIR) / 'darag_variant.json'
USE_RETRIEVAL = json.loads(vfile.read_text(encoding='utf-8')).get('use_retrieval', True) if vfile.exists() else True
print('base model:', BASE_MODEL, '| use_retrieval:', USE_RETRIEVAL)

tok_src = ADAPTER_DIR if (Path(ADAPTER_DIR) / 'tokenizer_config.json').exists() else BASE_MODEL
tok = AutoTokenizer.from_pretrained(tok_src, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                            device_map='auto', trust_remote_code=True)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()

SYS = ('You correct Vietnamese medical ASR transcripts. Preserve code-switched medical '
       'terms, numbers, units, acronyms, and clinical meaning. Use the other hypotheses '
       'and the retrieved named entities only as evidence. Output only the corrected transcript.')

def correct(raw_asr, terms=None):
    lines = [f'Best hypothesis: {raw_asr}']
    if USE_RETRIEVAL and terms:
        lines.append('Named entities: ' + ', '.join(terms))
    user = '\n'.join(lines)
    prompt = (f'<|im_start|>system\n{SYS}\n<|im_end|>\n'
              f'<|im_start|>user\n{user}\n<|im_end|>\n<|im_start|>assistant\n')
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                              num_beams=1, pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return text.split('<|im_end|>')[0].strip()

# Try a couple of deliberately mangled Vietnamese medical transcripts.
samples = [
    ('bệnh nhân đau ngực sp o hai chín tám phần trăm', ['SpO2']),
    ('TOTTE quá cao và DI HAIROTE tăng', ['testosterone', 'dihydrotestosterone']),
]
for raw, terms in samples:
    print('RAW      :', raw)
    print('CORRECTED:', correct(raw, terms))
    print()

# Your turn: correct("<your raw ASR text>", ["OptionalTerm"]) 
